# Coder: which free coding model fixes bugs best? (Ollama on Kaggle's free GPU)

**What it does:** downloads each model in `MODELS` with Ollama and gives it the 164 tasks of
**HumanEvalFix (Python)**: a small Python function with a bug in it. The model must send back the fixed
function. The fix is then run against the task's hidden tests: all pass = the task is fixed.
The score is the share of tasks fixed with one try (pass@1, greedy: temperature 0).

It saves everything to `coder_eval.json` (Output tab) after every model, so nothing is lost if it stops.
Time: about `MINUTES_PER_MODEL` minutes per model at most (download + answers), so about 3½ hours for 4 models.

**Settings (right side):** Accelerator **GPU T4 x2**, Internet **on**. Then **Run all**.

**Or with no clicks:** on GitHub, **Actions → Coder model on Kaggle → Run workflow**, or ask Claude.

The tasks: HumanEvalPack by BigCode (`bigcode/humanevalpack`, MIT licence).

In [ ]:
#@title 1. Settings: which models, how many tasks, time per model
import glob, json, os, re, shutil, subprocess, sys, time, urllib.request

# The models: names from ollama.com/library. Claude and GitHub Actions fill this line.
MODELS = ["qwen2.5-coder:7b", "qwen2.5-coder:14b", "qwen3-coder:30b", "gpt-oss:20b"]
# How many tasks to use (None = all 164). Claude and GitHub Actions fill this line.
LIMIT = None
MINUTES_PER_MODEL = 45  # a model stops after this (download, loading and answers), so the run ends in time
OPTIONS = {"temperature": 0, "num_ctx": 4096, "num_predict": 1024}  # temperature 0: the same answer every time
THINK = {"gpt-oss": "low"}  # thinking models: how much to think first ("low" fits 164 tasks in the time)
THINKING_NUM_PREDICT = 3072  # thinking models: room for the thinking and the answer (inside num_ctx)
# Downloads go to the big disk. Not /kaggle/working: it holds only about 20 GB, and all of it is saved as output.
MODELS_DIR = "/tmp/ollama-models"
DELETE_AFTER = True  # delete each model after its turn, so the disk holds only one at a time
WORKING_DIR = "/kaggle/working"  # coder_eval.json is saved here (the notebook's Output tab)

In [ ]:
#@title 2. Helpers (no need to read; built from coder/fixeval.py)
"""HumanEvalFix (Python) through Ollama: show a model a buggy function, run its fix against the hidden tests.

Plain Python (no packages to install). coder/build.py puts this whole file into coder_eval.ipynb, and
coder/run_eval.py uses table() to print the results, so the notebook and the report agree.
"""

import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request

BENCHMARK = "HumanEvalFix (Python)"
DATASET = "bigcode/humanevalpack"  # config "python", split "test": 164 tasks
LICENCE = "MIT"  # the HumanEvalPack licence (bigcode/humanevalpack on Hugging Face)
DATASET_URL = "https://huggingface.co/datasets/bigcode/humanevalpack/resolve/main/data/python/data/humanevalpack.jsonl"
NEEDED = ("task_id", "declaration", "buggy_solution", "test", "entry_point")
INSTRUCTION = "Fix bugs in {entry_point}. Reply with the full corrected function in one ```python block."
SETTING = "HumanEvalFix, no tests shown; greedy (temperature 0); pass@1 = tests pass with the one answer"
TEST_TIMEOUT = 10  # seconds for one task's tests
# What the standard HumanEval runners put above the code (standard library only).
IMPORTS = """import math
import re
import sys
import copy
import datetime
import itertools
import collections
import heapq
import functools
import hashlib
import string
from typing import *
from collections import *
"""


# ---- the tasks ----

def load_tasks():
    """All HumanEvalFix Python tasks (a list of dicts) and where they came from."""
    try:
        with urllib.request.urlopen(DATASET_URL, timeout=120) as resp:
            text = resp.read().decode("utf-8")
        rows, source = [json.loads(line) for line in text.splitlines() if line.strip()], DATASET_URL
    except Exception as first:  # noqa: BLE001 - try the datasets library next
        rows, source, problems = None, None, [f"{DATASET_URL}: {first!r}"]
        for revision in ("refs/convert/parquet", None):
            try:
                from datasets import load_dataset

                rows = [dict(r) for r in load_dataset(DATASET, "python", split="test", revision=revision)]
                source = f"datasets.load_dataset({DATASET!r}, 'python', split='test', revision={revision!r})"
                break
            except Exception as e:  # noqa: BLE001
                problems.append(f"datasets (revision {revision}): {e!r}")
        if rows is None:
            raise RuntimeError("Could not download HumanEvalFix: " + " | ".join(problems)) from None
    if not rows:
        raise RuntimeError(f"HumanEvalFix from {source} has no tasks")
    missing = [f for f in NEEDED if f not in rows[0]]
    if missing:
        raise RuntimeError(f"HumanEvalFix rows have no {missing}; they have {sorted(rows[0])}")
    return rows, source


def prompt_for(task):
    """The message to the model: the buggy function, then the instruction (no tests shown)."""
    buggy = (task["declaration"] + task["buggy_solution"]).strip("\n")
    return f"```python\n{buggy}\n```\n\n" + INSTRUCTION.format(entry_point=task["entry_point"])


# ---- the reply ----

THINKING = re.compile(r"<think(?:ing)?>.*?</think(?:ing)?>", re.DOTALL)
BLOCK = re.compile(r"```[ \t]*([\w+.-]*)[^\n]*\n(.*?)```", re.DOTALL)
OPEN_BLOCK = re.compile(r"```[ \t]*(?:python3?|py)?[ \t]*\n(.*)\Z", re.DOTALL)  # cut off before the closing ```


def without_thinking(reply):
    """The reply without <think>...</think> parts (a thinking model's notes before its answer)."""
    reply = THINKING.sub("", reply or "")
    if "</think>" in reply:  # the opening tag was in the prompt template, only the closing one came back
        reply = reply.rsplit("</think>", 1)[1]
    return reply


def extract_code(reply):
    """The code to test: the last ```python block (or the last block of any kind; or an unclosed one at the end;
    or else the whole reply)."""
    reply = without_thinking(reply)
    blocks = BLOCK.findall(reply)
    python = [body for lang, body in blocks if lang.lower() in ("python", "python3", "py", "")]
    if python or blocks:
        return (python or [body for _, body in blocks])[-1].strip("\n") + "\n"
    cut_off = OPEN_BLOCK.search(reply)
    return (cut_off.group(1) if cut_off else reply).strip("\n") + "\n"


def program_for(code, task):
    """The file that is run: imports, the model's code, the dataset's tests and the call that runs them."""
    test = task["test"]
    call = f"check({task['entry_point']})"
    if not re.search(rf"^check\(\s*{re.escape(task['entry_point'])}\s*\)", test, re.MULTILINE):
        test = test.rstrip("\n") + f"\n\n{call}\n"
    return "\n".join((IMPORTS, task.get("import") or "", code, "", test))


def run_program(program, timeout=None):
    """Run it in a new Python process. (passed, why): passed when it ends with exit code 0 within the time
    (TEST_TIMEOUT seconds unless timeout is given)."""
    timeout = timeout or TEST_TIMEOUT
    with tempfile.TemporaryDirectory() as folder:
        path = os.path.join(folder, "task.py")
        with open(path, "w", encoding="utf-8") as fh:
            fh.write(program)
        try:
            done = subprocess.run([sys.executable, path], cwd=folder, stdin=subprocess.DEVNULL, capture_output=True,
                                  text=True, timeout=timeout, check=False)
        except subprocess.TimeoutExpired:
            return False, f"timeout after {timeout} s"
    if done.returncode == 0:
        return True, "passed"
    lines = [line for line in (done.stderr or done.stdout).splitlines() if line.strip()]
    return False, (lines[-1].strip()[:200] if lines else f"exit code {done.returncode}")


# ---- Ollama ----

def post(url, path, body, timeout=600):
    request = urllib.request.Request(url + path, json.dumps(body).encode(), {"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as resp:
            return json.load(resp)
    except urllib.error.HTTPError as e:  # Ollama says why in the body: {"error": "..."}
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode('utf-8', 'replace')[:500]}") from None


def pull(url, model, log=print):
    """Download a model (like `ollama pull`). Its size in GB. Prints each new step."""
    request = urllib.request.Request(url + "/api/pull", json.dumps({"model": model, "stream": True}).encode(),
                                     {"Content-Type": "application/json"})
    sizes, last = {}, None
    try:
        with urllib.request.urlopen(request, timeout=900) as resp:
            for line in resp:
                if not line.strip():
                    continue
                step = json.loads(line)
                if step.get("error"):
                    raise RuntimeError(step["error"])
                if step.get("digest") and step.get("total"):
                    sizes[step["digest"]] = step["total"]
                status = re.sub(r"pulling [0-9a-f]{12,}", "pulling a part", step.get("status", ""))
                if status != last:
                    log(f"   {model}: {status}")
                    last = status
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode('utf-8', 'replace')[:500]}") from None
    if last != "success":
        raise RuntimeError(f"the download did not finish (last step: {last})")
    return round(sum(sizes.values()) / 1e9, 1)


def in_memory(url, model):
    """The model as Ollama holds it now (from /api/ps: size, size_vram, ...), or None when it is not loaded."""
    with urllib.request.urlopen(url + "/api/ps", timeout=10) as resp:
        models = json.load(resp).get("models") or []
    names = {model, model + ":latest"}
    return next((m for m in models if m.get("name") in names or m.get("model") in names), None)


def processor(url, model):
    """Where the model runs, as `ollama ps` says it: "100% GPU", "100% CPU", "25%/75% CPU/GPU", or None."""
    try:
        m = in_memory(url, model)
    except Exception:  # noqa: BLE001
        return None
    if not m or not m.get("size"):
        return None
    cpu = round(100 * (m["size"] - (m.get("size_vram") or 0)) / m["size"])
    return "100% GPU" if cpu <= 0 else "100% CPU" if cpu >= 100 else f"{cpu}%/{100 - cpu}% CPU/GPU"


def unload(url, model):
    """Take the model out of memory (keep_alive 0) and wait until it is gone."""
    post(url, "/api/generate", {"model": model, "keep_alive": 0}, timeout=120)
    for _ in range(60):
        if in_memory(url, model) is None:
            return
        time.sleep(1)


def delete(url, model):
    """Delete the downloaded model (like `ollama rm`), to free the disk for the next one."""
    request = urllib.request.Request(url + "/api/delete", json.dumps({"model": model}).encode(),
                                     {"Content-Type": "application/json"}, method="DELETE")
    with urllib.request.urlopen(request, timeout=120):
        pass


def text_of(command):
    """What a command printed ("" when it is missing or fails), e.g. `ollama ps` or nvidia-smi."""
    try:
        return subprocess.run(command, shell=True, capture_output=True, text=True, timeout=60,
                              check=False).stdout.strip()
    except Exception:  # noqa: BLE001
        return ""


def think_for(model, think):
    """The "think" setting for this model ({"gpt-oss": "low"} → "low" for gpt-oss:20b), or None."""
    return next((level for prefix, level in (think or {}).items() if model.startswith(prefix)), None)


def ask(url, model, prompt, options, think=None, timeout=600):
    """One chat request (no system message). The final answer only: a thinking model's thinking is kept apart."""
    body = {"model": model, "messages": [{"role": "user", "content": prompt}], "stream": False, "options": options,
            "keep_alive": "30m", **({"think": think} if think else {})}
    started = time.time()
    data = post(url, "/api/chat", body, timeout=timeout)
    message = data.get("message") or {}
    return {"reply": message.get("content") or "", "thinking_chars": len(message.get("thinking") or ""),
            "seconds": round(time.time() - started, 2), "tokens": data.get("eval_count") or 0,
            "token_seconds": (data.get("eval_duration") or 0) / 1e9, "done_reason": data.get("done_reason")}


# ---- one model ----

def disk_free_gb(path):
    try:
        return round(shutil.disk_usage(path).free / 1e9, 1)
    except OSError:
        return None


def evaluate_model(url, model, tasks, entry, minutes, options, think=None, thinking_num_predict=None,
                   save=lambda: None, log=print, models_dir="/tmp"):
    """Download, load and score one model, filling entry (a dict in the results) as it goes, so a failure part-way
    keeps everything done until then. Stops when `minutes` have passed (counting the download and loading)."""
    deadline = time.time() + minutes * 60
    level = think_for(model, think)
    options = {**options, **({"num_predict": thinking_num_predict} if level and thinking_num_predict else {})}
    entry.update(model=model, think=level, options=options, total=len(tasks), done=0, passed=0, pass_at_1=None,
                 rate_done=None, pulled=False, pull_seconds=None, size_gb=None, load_seconds=None, processor=None,
                 ollama_ps=None, gpu_memory=None, avg_seconds=None, tokens_per_s=None, cut_off=0, no_code_block=0,
                 stopped_by_time=False, minutes=None, disk_free_gb_before=disk_free_gb(models_dir), errors=[],
                 tasks=[])
    started = time.time()

    log(f"\n== {model}: download")
    entry["size_gb"] = pull(url, model, log)
    entry.update(pulled=True, pull_seconds=round(time.time() - started, 1))
    log(f"   {model}: {entry['size_gb']} GB in {entry['pull_seconds']} s")
    save()

    loading = time.time()
    post(url, "/api/generate", {"model": model, "options": options, "keep_alive": "30m"}, timeout=900)  # only load
    entry["load_seconds"] = round(time.time() - loading, 1)
    log(f"   {model}: loaded in {entry['load_seconds']} s")
    save()

    failures_in_a_row = 0
    for i, task in enumerate(tasks, 1):
        if time.time() > deadline:
            entry["stopped_by_time"] = True
            log(f"   {model}: stopped at {minutes} minutes, after {entry['done']} of {len(tasks)} tasks")
            break
        record = {"task_id": task["task_id"], "passed": False}
        try:
            answer = ask(url, model, prompt_for(task), options, level)
        except Exception as e:  # noqa: BLE001 - count it as a fail and go on
            record["why"] = f"request failed: {e!r}"[:300]
            entry["errors"].append(f"{task['task_id']}: {record['why']}")
            failures_in_a_row += 1
        else:
            failures_in_a_row = 0
            code = extract_code(answer["reply"])
            has_block = "```" in without_thinking(answer["reply"])
            passed, why = run_program(program_for(code, task))
            if not answer["reply"].strip():
                why = "no answer" + (" (cut off while thinking)" if answer["done_reason"] == "length" else "")
            record.update(answer, passed=passed, why=why, code_block=has_block)
            entry["cut_off"] += answer["done_reason"] == "length"
            entry["no_code_block"] += not has_block
        entry["tasks"].append(record)
        entry["done"] += 1
        entry["passed"] += record["passed"]
        log(f"   {i:3d}/{len(tasks)} {task['task_id']:16s} {'pass' if record['passed'] else 'FAIL'}  "
            f"{record.get('seconds', '-')} s  {'' if record['passed'] else record['why'][:80]}")
        if i == 1:  # where it runs, once it has answered (`ollama ps`)
            entry.update(processor=processor(url, model), ollama_ps=text_of("ollama ps") or None,
                         gpu_memory=text_of("nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader")
                         .replace("\n", "; ") or None)
            log(f"   {model} runs on: {entry['processor']}")
        summarise(entry, started)
        if i % 10 == 0:
            save()
        if failures_in_a_row >= 3:
            entry["errors"].append("3 requests in a row failed: stopped this model")
            log(f"   {model}: 3 requests in a row failed, going on to the next model")
            break
    summarise(entry, started)
    save()
    return entry


def summarise(entry, started):
    """The totals of one model from its task records."""
    answered = [t for t in entry["tasks"] if "seconds" in t]
    token_seconds = sum(t["token_seconds"] for t in answered)
    entry["pass_at_1"] = round(entry["passed"] / entry["total"], 4) if entry["total"] and \
        entry["done"] == entry["total"] else None
    entry["rate_done"] = round(entry["passed"] / entry["done"], 4) if entry["done"] else None
    entry["avg_seconds"] = round(sum(t["seconds"] for t in answered) / len(answered), 2) if answered else None
    entry["tokens_per_s"] = round(sum(t["tokens"] for t in answered) / token_seconds, 1) if token_seconds else None
    entry["minutes"] = round((time.time() - started) / 60, 1)


def tidy_up(url, model, entry, delete_after=True, log=print):
    """Unload the model (and delete it, to free the disk). Problems are recorded, never raised."""
    for step, do in (("unload", unload), ("delete", delete if delete_after else None)):
        if do is None:
            continue
        try:
            do(url, model)
        except Exception as e:  # noqa: BLE001 - the next model must still run
            entry.setdefault("errors", []).append(f"{step}: {e!r}"[:300])
            log(f"   {model}: {step} failed: {e!r}")


# ---- the report ----

def table(results):
    """The results as text lines: one row per model."""
    lines = [f"{'model':24s} {'passed':>13s} {'%':>5s} {'s/task':>7s} {'tok/s':>6s}  {'ran on':16s} note"]
    for m in results.get("models") or []:
        done, total = m.get("done") or 0, m.get("total") or 0
        rate = m.get("pass_at_1") if m.get("pass_at_1") is not None else m.get("rate_done")
        note = []
        if not m.get("pulled"):
            note.append("download failed")
        elif m.get("load_seconds") is None:
            note.append("did not load")
        if m.get("stopped_by_time"):
            note.append(f"stopped at the time limit after {done} of {total} tasks")
        elif done and done < total:
            note.append(f"stopped after {done} of {total} tasks")
        if m.get("cut_off"):
            note.append(f"{m['cut_off']} cut off")
        if m.get("think"):
            note.append(f"think {m['think']}")
        if m.get("errors"):
            note.append(f"{len(m['errors'])} error" + ("s" if len(m["errors"]) > 1 else ""))
        passed = f"{m.get('passed') or 0}/{done}" + ("" if done == total else f" of {total}")
        percent = "-" if rate is None else f"{rate:.0%}"
        lines.append(f"{m.get('model', '?'):24s} {passed:>13s} {percent:>5s} {m.get('avg_seconds') or '-':>7} "
                     f"{m.get('tokens_per_s') or '-':>6}  {m.get('processor') or '-':16s} {'; '.join(note)}".rstrip())
    lines.append("(% = passed of all tasks when all were done, else of the tasks done; greedy pass@1)")
    return lines


RESULTS_FILE = 'coder_eval.json'

RESULTS = {"benchmark": BENCHMARK, "dataset": DATASET, "config": "python", "split": "test", "licence": LICENCE,
           "setting": SETTING, "instruction": INSTRUCTION, "test_timeout_s": TEST_TIMEOUT, "options": OPTIONS,
           "think": THINK, "thinking_num_predict": THINKING_NUM_PREDICT, "minutes_per_model": MINUTES_PER_MODEL,
           "limit": LIMIT, "models_asked": MODELS, "models_dir": MODELS_DIR, "ollama_version": None, "gpu": None,
           "gpus": None, "disk": None, "dataset_source": None, "fields": None, "tasks_total": None, "tasks_used": None,
           "models": [], "errors": []}
OLLAMA_LOG = "/tmp/ollama.log"  # what `ollama serve` prints (cell 3)


def save_results():
    """Write RESULTS to coder_eval.json. Done after every stage, so a stop part-way loses nothing."""
    if os.path.isdir(WORKING_DIR):
        with open(os.path.join(WORKING_DIR, "coder_eval.json"), "w", encoding="utf-8") as fh:
            json.dump(RESULTS, fh, ensure_ascii=False, indent=1)


def failed(stage, error):
    """Record a failed stage and go on (Kaggle's Run all stops at the first error that is not caught)."""
    message = f"{stage}: {error!r}" if isinstance(error, Exception) else f"{stage}: {error}"
    print("FAILED", message)
    RESULTS["errors"].append(message)
    save_results()


def sh(command, timeout=120):
    """What a shell command printed, or "" when the command is missing or fails."""
    try:
        return subprocess.run(command, shell=True, capture_output=True, text=True, timeout=timeout).stdout.strip()
    except Exception:
        return ""

In [ ]:
#@title 3. Install Ollama and start it (about 1–2 minutes)
os.makedirs(MODELS_DIR, exist_ok=True)
os.environ["OLLAMA_MODELS"] = MODELS_DIR  # where `ollama serve` keeps the downloads
os.environ["OLLAMA_NUM_PARALLEL"] = "1"  # one question at a time: less GPU memory

OLLAMA_URL = "http://127.0.0.1:11434"


def ollama_version():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/api/version", timeout=5) as resp:
            return json.load(resp).get("version")
    except Exception:
        return None


if not ollama_version():
    if not shutil.which("ollama"):
        # zstd unpacks Ollama's download (.tar.zst). The download already holds the CUDA libraries; lspci is
        # left out on purpose: with it (and no nvidia-smi) install.sh would try to install NVIDIA drivers.
        if not shutil.which("zstd"):
            sh("apt-get update -qq; DEBIAN_FRONTEND=noninteractive apt-get install -y -qq zstd", timeout=600)
        install = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True,
                                 capture_output=True, text=True, timeout=1200)
        print("\n".join((install.stdout + install.stderr).replace("\r", "\n").splitlines()[-15:]))
        if not shutil.which("ollama"):
            failed("install Ollama", (install.stdout + install.stderr)[-1500:])
            raise RuntimeError("Ollama did not install (see above)")
    print(sh("ollama --version 2>&1"))
    log = open(OLLAMA_LOG, "w")  # stays open: the server writes to it
    server = subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"},
                              stdout=log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_version() or server.poll() is not None:
            break
        time.sleep(1)
if not ollama_version():
    tail = sh(f"tail -n 40 {OLLAMA_LOG}")
    print(tail)
    failed("start Ollama", tail[-1500:])
    raise RuntimeError("Ollama did not start (its log is above)")
RESULTS["ollama_version"] = ollama_version()
RESULTS["gpu"] = (sh("nvidia-smi --query-gpu=name --format=csv,noheader").splitlines() or [None])[0]
print("Ollama", RESULTS["ollama_version"], "is running. GPU:", RESULTS["gpu"])
save_results()

RESULTS["gpus"] = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader") or None
RESULTS["disk"] = sh(f"df -h {MODELS_DIR} {WORKING_DIR} 2>/dev/null") or None
print("GPUs:", RESULTS["gpus"])
print(RESULTS["disk"])
save_results()

In [ ]:
#@title 4. The tasks: HumanEvalFix (Python), buggy functions with hidden tests
try:
    ALL_TASKS, RESULTS["dataset_source"] = load_tasks()
except Exception as e:
    failed("download the tasks", e)
    raise
RESULTS["fields"] = sorted(ALL_TASKS[0])
TASKS = ALL_TASKS[:LIMIT] if LIMIT else ALL_TASKS
RESULTS.update(tasks_total=len(ALL_TASKS), tasks_used=len(TASKS))
print(f"{len(ALL_TASKS)} tasks from {RESULTS['dataset_source']}; using {len(TASKS)}")
print("Fields:", ", ".join(RESULTS["fields"]))
for field in RESULTS["fields"]:  # the first task, field by field
    print(f"--- {field}:\n{str(TASKS[0][field])[:400]}")
print("--- What a model is sent:\n" + prompt_for(TASKS[0]))
save_results()

In [ ]:
#@title 5. Each model: download, load, fix every task, run the tests
for model in MODELS:
    entry = {"model": model}
    RESULTS["models"].append(entry)
    try:
        evaluate_model(OLLAMA_URL, model, TASKS, entry, MINUTES_PER_MODEL, OPTIONS, THINK, THINKING_NUM_PREDICT,
                       save=save_results, models_dir=MODELS_DIR)
    except Exception as e:  # record it and go on to the next model
        entry.setdefault("errors", []).append(f"{e!r}"[:500])
        failed(model, e)
    if entry.get("pulled"):
        tidy_up(OLLAMA_URL, model, entry, DELETE_AFTER)
    else:  # a download that stopped part-way leaves its pieces: remove them, so the next model has the disk
        for part in glob.glob(os.path.join(MODELS_DIR, "blobs", "*-partial*")):
            os.remove(part)
    save_results()
    print("\n".join(table(RESULTS)))

In [ ]:
#@title 6. Results
print("\n".join(table(RESULTS)))
for m in RESULTS["models"]:
    for error in m.get("errors") or []:
        print(f"{m['model']}: {error}")
for error in RESULTS["errors"]:
    print("Error:", error)
save_results()
print("Saved:", os.path.join(WORKING_DIR, RESULTS_FILE))

## What next
The best model here is the one to train next (phase 2), if it runs fast enough on this GPU.
`coder_eval.json` keeps every reply, so the failed tasks can be read later.